# 05 모니터링 스토리 및 대시보드


In [ ]:
import os
from pathlib import Path
import warnings

# 결과물 폴더 고정
PROJECT_DIR = Path("/Users/chankyulee/Desktop/ModuLABS/05_TimeSeries/Projects/FTS_Projects")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "05_monitoring_story_and_dashboard"
PREV_OUTPUT_DIR = PROJECT_DIR / "outputs" / "04_threshold_tuning_and_rules"
MPLCONFIG_DIR = PROJECT_DIR / ".mplconfig"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
sns.set_theme(style="whitegrid", context="notebook")

print(f"PREV_OUTPUT_DIR exists: {PREV_OUTPUT_DIR.exists()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


- 이번 노트북은 앞선 결과를 읽어와 스토리와 대시보드 관점으로 정리하는 단계
- 분석 산출물과 발표용 정리 자료를 분리해두면 마지막 패키징 단계가 훨씬 수월함


## 1. 최종 규칙과 KPI 불러오기

- 최종 rule과 test KPI 재정리
- baseline 대비 개선 폭 확인


In [ ]:
# 이전 노트북 결과 재사용
selected_configs = pd.read_csv(PREV_OUTPUT_DIR / "selected_rule_configs.csv")
test_rule_comparison = pd.read_csv(PREV_OUTPUT_DIR / "test_rule_comparison.csv")
test_rule_predictions = pd.read_csv(PREV_OUTPUT_DIR / "test_rule_predictions.csv", parse_dates=["time"])
rule_window_sample = pd.read_csv(PREV_OUTPUT_DIR / "rule_window_sample.csv", parse_dates=["time"])

# baseline 대비 개선 폭 계산
baseline_row = test_rule_comparison.loc[
    test_rule_comparison["config_name"] == "baseline_zscore"
].iloc[0]

monitoring_kpi_summary = test_rule_comparison.copy()
monitoring_kpi_summary["point_f1_delta_vs_baseline"] = (
    monitoring_kpi_summary["point_f1"] - baseline_row["point_f1"]
)
monitoring_kpi_summary["point_fpr_reduction_vs_baseline"] = (
    1 - monitoring_kpi_summary["point_fpr"] / baseline_row["point_fpr"]
)
monitoring_kpi_summary["false_alerts_per_day_reduction_vs_baseline"] = (
    1 - monitoring_kpi_summary["false_alerts_per_day"] / baseline_row["false_alerts_per_day"]
)
monitoring_kpi_summary["event_f1_delta_vs_baseline"] = (
    monitoring_kpi_summary["event_f1"] - baseline_row["event_f1"]
)

# 운영 기본안 / 보수안 역할 부여
monitoring_kpi_summary["operating_role"] = monitoring_kpi_summary["config_name"].map(
    {
        "baseline_zscore": "비교 기준",
        "balanced_selected": "기본 운영안",
        "conservative_selected": "보수 운영안",
    }
)

monitoring_kpi_summary.to_csv(OUTPUT_DIR / "monitoring_kpi_summary.csv", index=False)

display(selected_configs)
display(monitoring_kpi_summary)


- `balanced_selected`는 baseline 대비 point F1이 좋아지고, point FPR과 false alert/day를 크게 감소
- `conservative_selected`는 가장 조용하지만 event_f1은 더 내려감
- 따라서 기본 추천안은 balanced, 보수 운영안은 conservative가 자연스럽다


## 2. Daily Monitoring Summary

- 하루 단위로 이벤트 수와 경보 수 정리
- 운영 누락 여부 확인


In [ ]:
# 하루 단위 집계로 재정리
test_rule_predictions["date"] = test_rule_predictions["time"].dt.date

daily_monitoring_summary = (
    test_rule_predictions
    .groupby("date")[["pseudo_event", "baseline_zscore", "balanced_selected", "conservative_selected"]]
    .sum()
    .reset_index()
)

# 이벤트가 있던 날 경보가 0개였는지 확인
daily_monitoring_summary["balanced_missed_day"] = (
    (daily_monitoring_summary["pseudo_event"] > 0)
    & (daily_monitoring_summary["balanced_selected"] == 0)
)
daily_monitoring_summary["conservative_missed_day"] = (
    (daily_monitoring_summary["pseudo_event"] > 0)
    & (daily_monitoring_summary["conservative_selected"] == 0)
)

top_balanced_alert_days = (
    daily_monitoring_summary
    .sort_values("balanced_selected", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

daily_monitoring_summary.to_csv(OUTPUT_DIR / "daily_monitoring_summary.csv", index=False)
top_balanced_alert_days.to_csv(OUTPUT_DIR / "top_balanced_alert_days.csv", index=False)

daily_kpi = pd.DataFrame(
    {
        "metric": [
            "num_days",
            "avg_daily_pseudo_event",
            "avg_daily_baseline_alert",
            "avg_daily_balanced_alert",
            "avg_daily_conservative_alert",
            "max_daily_balanced_alert",
            "balanced_missed_days",
            "conservative_missed_days",
        ],
        "value": [
            len(daily_monitoring_summary),
            daily_monitoring_summary["pseudo_event"].mean(),
            daily_monitoring_summary["baseline_zscore"].mean(),
            daily_monitoring_summary["balanced_selected"].mean(),
            daily_monitoring_summary["conservative_selected"].mean(),
            daily_monitoring_summary["balanced_selected"].max(),
            int(daily_monitoring_summary["balanced_missed_day"].sum()),
            int(daily_monitoring_summary["conservative_missed_day"].sum()),
        ],
    }
)

display(daily_kpi)
display(top_balanced_alert_days)


- test 204일 기준으로 balanced의 일평균 경보 수가 실제 pseudo event 규모와 가장 가깝게 내려옴
- balanced는 이벤트가 있던 날을 놓친 경우가 없고, conservative는 일부 약한 날을 놓침
- daily summary 기준으로도 balanced가 기본 운영안에 더 적합


## 3. Dashboard Mockup

- KPI 비교
- 일별 경보 흐름
- 상위 경보일과 이벤트 예시 창 정리


In [ ]:
# 한 화면에 KPI + 흐름 + 예시 이벤트 배치
fig = plt.figure(figsize=(18, 13))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.1])

# 1) KPI 비교
ax1 = fig.add_subplot(gs[0, 0])
kpi_plot = monitoring_kpi_summary[["config_name", "point_f1", "event_f1", "false_alerts_per_day"]].copy()
kpi_plot = kpi_plot.set_index("config_name")
kpi_plot[["point_f1", "event_f1"]].plot(kind="bar", ax=ax1)
ax1.set_title("Rule KPI Comparison")
ax1.tick_params(axis="x", rotation=20)
ax1.set_ylabel("score")

# false alert/day는 보조 축 사용
ax1b = ax1.twinx()
ax1b.plot(range(len(kpi_plot)), kpi_plot["false_alerts_per_day"], color="black", marker="o", linewidth=2)
ax1b.set_ylabel("false alerts / day")

# 2) 일별 모니터링 추이
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(daily_monitoring_summary["date"], daily_monitoring_summary["pseudo_event"], label="pseudo_event", color="red", linewidth=1.2)
ax2.plot(daily_monitoring_summary["date"], daily_monitoring_summary["balanced_selected"], label="balanced_alert", color="royalblue", linewidth=1.2)
ax2.plot(daily_monitoring_summary["date"], daily_monitoring_summary["conservative_selected"], label="conservative_alert", color="green", linewidth=1.2)
ax2.set_title("Daily Monitoring Summary")
ax2.tick_params(axis="x", rotation=30)
ax2.legend()

# 3) 경보가 많았던 날짜
ax3 = fig.add_subplot(gs[1, 0])
top_plot = top_balanced_alert_days.copy()
top_plot["date"] = top_plot["date"].astype(str)
sns.barplot(data=top_plot, x="date", y="balanced_selected", color="royalblue", ax=ax3)
ax3.set_title("Top 10 Balanced Alert Days")
ax3.tick_params(axis="x", rotation=35)
ax3.set_ylabel("alert count")

# 4) 예시 이벤트 창
ax4 = fig.add_subplot(gs[1, 1])
window_plot = rule_window_sample.fillna(0).copy()
ax4.plot(window_plot["time"], window_plot["raw_close"], color="black", linewidth=1.0, label="close")
ax4.scatter(window_plot.loc[window_plot["pseudo_event"] == 1, "time"], window_plot.loc[window_plot["pseudo_event"] == 1, "raw_close"], color="red", s=60, label="pseudo_event")
ax4.scatter(window_plot.loc[window_plot["balanced_selected"] == 1, "time"], window_plot.loc[window_plot["balanced_selected"] == 1, "raw_close"], color="royalblue", s=35, label="balanced")
ax4.scatter(window_plot.loc[window_plot["conservative_selected"] == 1, "time"], window_plot.loc[window_plot["conservative_selected"] == 1, "raw_close"], color="green", s=35, label="conservative")
ax4.set_title("Example Event Window")
ax4.tick_params(axis="x", rotation=30)
ax4.legend()

plt.tight_layout()
plt.show()


- dashboard 기준으로도 balanced는 baseline보다 경보량을 줄이면서 KPI를 크게 해치지 않는 쪽에 위치
- conservative는 가장 조용하지만 이벤트 반응도 함께 줄어듦
